# GEPS要素清单：112-130

In [ ]:
from reki import from_source

In [ ]:
from data_notebook.data import get_data_file

path = get_data_file("cma-geps")
source = from_source({"kind": "file", "path": str(path)})
path

In [ ]:
def check_message(ordinal, field, *other_fields):
    location = field.metadata["source"]
    assert location["ordinal"] == ordinal

    with path.open("rb") as original:
        original.seek(location["offset"])
        message = original.read(location["length"])

    for selected in (field, *other_fields):
        assert selected.metadata["source"] == location
        assert selected.read_grib_message() == message

## 等压面：相对湿度

### 层次

17 层列表（百帕 (hPa)）：

- 1000
- 975
- 950
- 925
- 900
- 850
- 800
- 750
- 700
- 650
- 600
- 550
- 500
- 400
- 300
- 200
- 100

### 相对湿度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 相对湿度 | rh | RH | r | (0, 1, 1) | pl | 17层 | 112-128 |

In [ ]:
# 按 CedarKit 要素名
fields = source.sel(
    parameter="rh",
    member=1,
    level_type="pressure",
    level=[100, 550, 850, 1000],
    level_unit="hPa",
).all()

# 按 ecCodes 短名
by_short_name = source.sel(
    native={"grib": {"shortName": "r"}},
    level_type="pressure",
    level=[100, 550, 850, 1000],
    level_unit="hPa",
).all()

# 按 GRIB2 原生键
by_grib = source.sel(native={"grib": {
    "discipline": 0,
    "parameterCategory": 1,
    "parameterNumber": 1,
    "typeOfFirstFixedSurface": 100,
    "scaleFactorOfFirstFixedSurface": 0,
    "scaledValueOfFirstFixedSurface": [10000, 55000, 85000, 100000],
    "typeOfEnsembleForecast": 3,
    "perturbationNumber": 1,
}}).all()

for ordinal, field, short_name_field, grib_field in zip(
    [112, 117, 123, 128], fields, by_short_name, by_grib, strict=True,
):
    check_message(ordinal, field, short_name_field, grib_field)

## 阵风与能见度

### 阵风风速

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 阵风风速 | gust | GUST | i10fg | (0, 2, 22) | heightAboveGround | 10 | 129 |

In [ ]:
# 按 CedarKit 要素名
field = source.sel(parameter="gust", member=1).one()

# 按 ecCodes 短名
by_short_name = source.sel(native={"grib": {"shortName": "i10fg"}}).one()

# 按 GRIB2 原生键
by_grib = source.sel(native={"grib": {
    "discipline": 0,
    "parameterCategory": 2,
    "parameterNumber": 22,
    "typeOfFirstFixedSurface": 103,
    "scaleFactorOfFirstFixedSurface": 0,
    "scaledValueOfFirstFixedSurface": 10,
    "typeOfEnsembleForecast": 3,
    "perturbationNumber": 1,
}}).one()

check_message(129, field, by_short_name, by_grib)

### 能见度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 能见度 | vis | VIS | vis | (0, 19, 0) | surface | - | 130 |

In [ ]:
# 按 CedarKit 要素名
field = source.sel(parameter="vis", member=1).one()

# 按 ecCodes 短名
by_short_name = source.sel(native={"grib": {"shortName": "vis"}}).one()

# 按 GRIB2 原生键
by_grib = source.sel(native={"grib": {
    "discipline": 0,
    "parameterCategory": 19,
    "parameterNumber": 0,
    "typeOfFirstFixedSurface": 1,
    "typeOfEnsembleForecast": 3,
    "perturbationNumber": 1,
}}).one()

check_message(130, field, by_short_name, by_grib)

In [ ]:
source.close()